Yes. Before we continue, you **should understand these three**, because they are the foundation of professional preprocessing.

You have:

```text
37 numerical columns
43 categorical columns
= 80 features
```

We need to preprocess these two groups differently.

---

# 1. What is `Pipeline`?

Think of a `Pipeline` as a **sequence of operations performed automatically in order**.

For example, numerical data:

```text
Raw numerical data
       ↓
Fill missing values
       ↓
Scale features
       ↓
Ready for model
```

Instead of doing:

```python
X = imputer.fit_transform(X)
X = scaler.fit_transform(X)
model.fit(X, y)
```

we create:

```python
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
```

Now:

```python
numeric_pipeline.fit_transform(X_train)
```

automatically performs:

```text
1. Fit imputer on X_train
2. Transform X_train using imputer
3. Fit scaler
4. Transform using scaler
```

### Why is this useful?

Because in a real ML project, we don't want preprocessing scattered throughout our code.

We want:

```text
preprocessing + model
```

to behave like **one reproducible system**.

---

# 2. What is `SimpleImputer`?

`SimpleImputer` handles missing values.

For example:

```text
Age
---
20
25
NaN
30
```

If we use:

```python
SimpleImputer(strategy="median")
```

it calculates:

```text
median = 25
```

and transforms:

```text
20
25
NaN  → 25
30
```

### Available common strategies

```python
SimpleImputer(strategy="mean")
```

→ replace missing numerical values with mean.

```python
SimpleImputer(strategy="median")
```

→ replace with median.

```python
SimpleImputer(strategy="most_frequent")
```

→ replace with most common category.

```python
SimpleImputer(strategy="constant", fill_value="Unknown")
```

→ replace with a fixed value.

---

## Why are we using it when our dataset has 0 missing values?

Excellent question.

Currently:

```python
df.isnull().sum().sum()
# 0
```

So **we don't technically need it for this particular cleaned dataset**.

But we're building a reusable ML pipeline.

Imagine tomorrow you receive:

```text
new_house_data.csv
```

and one house has:

```text
GarageArea = NaN
```

Our pipeline can automatically handle it.

More importantly, `SimpleImputer` inside the pipeline prevents us from accidentally calculating imputation statistics using the test set.

That's related to **data leakage**.

---

# 3. What is `ColumnTransformer`?

This is the most important one for your current dataset.

We have:

```text
37 numerical
43 categorical
```

But numerical and categorical data need **different operations**.

We want:

```text
                     ┌── Numerical → Imputer → Scaler ──┐
80 columns ──────────┤                                   ├──→ Model
                     └── Categorical → Imputer → OneHot ─┘
```

That's exactly what `ColumnTransformer` does.

---

## Example

We create:

```python
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])
```

Let's break it down.

### First:

```python
("num", numeric_transformer, numeric_features)
```

means:

> Take these 37 numerical columns and apply `numeric_transformer`.

Our numerical transformer is:

```python
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
```

So:

```text
37 numerical columns
       ↓
Median imputation
       ↓
StandardScaler
```

---

### Second:

```python
("cat", categorical_transformer, categorical_features)
```

means:

> Take these 43 categorical columns and apply `categorical_transformer`.

Our categorical transformer is:

```python
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
```

So:

```text
43 categorical columns
       ↓
Fill missing categories
       ↓
One-Hot Encoding
```

---

# 4. Why can't we just use one transformer?

Suppose we have:

```text
OverallQual = 8
Neighborhood = "NoRidge"
```

We can scale:

```text
OverallQual → StandardScaler
```

But doing:

```text
Neighborhood → StandardScaler
```

doesn't make sense.

Likewise, doing:

```text
OverallQual → OneHotEncoder
```

may not be what we want for our numerical pipeline.

Therefore:

```text
Numerical → numerical preprocessing
Categorical → categorical preprocessing
```

And `ColumnTransformer` lets us do both **simultaneously**.

---

# 5. The entire thing together

Our architecture becomes:

```text
                    X_train
                       │
                       ▼
              ColumnTransformer
                 /           \
                /             \
       Numerical              Categorical
          37                     43
           │                      │
           ▼                      ▼
      SimpleImputer         SimpleImputer
       (median)             (most frequent)
           │                      │
           ▼                      ▼
    StandardScaler          OneHotEncoder
           │                      │
           └──────────┬───────────┘
                      ▼
              Processed features
                      │
                      ▼
                   Model
```

That's the fundamental architecture we'll use.

---

# 6. VERY IMPORTANT: `fit` vs `transform`

This is something you absolutely need to understand.

Suppose training data has:

```text
LotArea:
1000
2000
3000
```

The scaler learns:

```text
mean
standard deviation
```

from **training data**.

```python
preprocessor.fit(X_train)
```

Then we transform:

```python
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)
```

Notice:

### Training

```python
fit_transform(X_train)
```

### Test

```python
transform(X_test)
```

**Never:**

```python
fit_transform(X_test)
```

because then the test data influences preprocessing.

That's **data leakage**.

---

# 7. And eventually we'll combine preprocessing + model

For example:

```python
model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])
```

Then simply:

```python
model_pipeline.fit(X_train, y_train)
```

Internally:

```text
X_train
   ↓
ColumnTransformer
   ↓
Numerical preprocessing
Categorical preprocessing
   ↓
Processed X_train
   ↓
Linear Regression
   ↓
learn
```

And prediction:

```python
model_pipeline.predict(X_test)
```

automatically does:

```text
X_test
   ↓
same preprocessing learned from train
   ↓
model
   ↓
prediction
```

This is **much safer and cleaner** than manually preprocessing everything.

---

## One correction for our project

Because we're going to compare **Linear Regression vs Random Forest vs Gradient Boosting vs XGBoost/LightGBM**, we shouldn't blindly use exactly the same preprocessing for every model.

For example:

| Model             | Scaling?       | One-Hot? |
| ----------------- | -------------- | -------- |
| Linear Regression | Usually useful | Yes      |
| KNN               | **Yes**        | Yes      |
| Decision Tree     | Not required   | Yes      |
| Random Forest     | Not required   | Yes      |
| Gradient Boosting | Not required   | Yes      |
| XGBoost           | Not required   | Yes      |
| LightGBM          | Not required   | Yes      |

We'll build **model-appropriate pipelines** rather than pretending one preprocessing setup is universally optimal.

---

### For now, don't run anything.

You now understand:

```text
Pipeline
   ↓
sequence of preprocessing/model steps

SimpleImputer
   ↓
handles missing values

ColumnTransformer
   ↓
applies different preprocessing to different column groups
```

**Next step:** we'll build the actual `preprocessor`, inspect the transformed dataset (including how 43 categorical columns become hundreds of encoded features), and then establish our **first baseline model**.
